# Qwen3.5-0.8B × TinyCeNN — Integrated Memory V1

This experiment tests bounded TinyCeNN memory on the **text backbone** of `Qwen/Qwen3.5-0.8B`.

Qwen3.5-0.8B uses 24 text decoder layers:
- 18 native `linear_attention` / Gated DeltaNet layers — left unchanged.
- 6 `full_attention` layers at **3, 7, 11, 15, 19, 23**.

Candidates:
- **conservative**: replace full-attention layers `3,23`
- **expanded**: replace **all six** full-attention layers `3,7,11,15,19,23`
- matched Transformer-readout controls use the same layer sets
- validation NLL selects only among the CeNN candidates; held-out test data is not used for selection

The Qwen3.5 post-attention output gate is preserved exactly. TinyCeNN's readout is intentionally **not fused** through that gate.


In [ ]:
import os, sys, json, subprocess, tempfile, shutil
from pathlib import Path
from datetime import datetime, timezone

REPO = Path(tempfile.mkdtemp(prefix="qwen35-cenn-")) / "TinyCeNN-LM"
subprocess.run(["git","clone","--quiet","https://github.com/vtavakkoli/TinyCeNN-LM.git",str(REPO)],check=True)
subprocess.run(["git","checkout","main"],cwd=REPO,check=True)
subprocess.run(["git","fetch","origin","main"],cwd=REPO,check=True)
subprocess.run(["git","reset","--hard","origin/main"],cwd=REPO,check=True)

# Qwen3.5 support is pinned to the tested current Transformers release.
subprocess.run([
    sys.executable,"-m","pip","install","-q",
    "transformers==5.17.0","datasets>=3,<6","pytest","pandas","matplotlib"
],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(REPO),"--no-deps"],check=True)

os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO),str(REPO/"src")])
sys.path[:0] = [str(REPO),str(REPO/"src")]

import torch, transformers
from transformers import AutoConfig

MODEL_ID = "Qwen/Qwen3.5-0.8B"
cfg = AutoConfig.from_pretrained(MODEL_ID).get_text_config(decoder=True)
FULL = [i for i,t in enumerate(cfg.layer_types) if t == "full_attention"]
LINEAR = [i for i,t in enumerate(cfg.layer_types) if t == "linear_attention"]
SOURCE = subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO,text=True).strip()

print("Source:", SOURCE)
print("Transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Text architecture:", cfg.model_type)
print("Text layers:", cfg.num_hidden_layers)
print("Linear-attention layers:", len(LINEAR), LINEAR)
print("Full-attention layers:", len(FULL), FULL)
print("Head dim:", cfg.head_dim, "heads:", cfg.num_attention_heads, "KV heads:", cfg.num_key_value_heads)

assert transformers.__version__ == "5.17.0"
assert cfg.model_type == "qwen3_5_text"
assert cfg.num_hidden_layers == 24
assert FULL == [3,7,11,15,19,23]
assert len(LINEAR) == 18


## Configuration

Start with `smoke` if you want a fast compatibility check. `balanced` is the default meaningful experiment.


In [ ]:
PROFILE = "balanced" # @param ["smoke","balanced","extended"]
SAVE_TO_DRIVE = True # @param {type:"boolean"}

PROFILES = {
    "smoke": dict(
        train_contexts="64,128", test_contexts="64,128,256",
        block_size=16, features=32,
        train_documents=3, validation_documents=2, test_documents=2, warm_documents=2,
        warm_steps=2, joint_steps=4, eval_every=2,
        timing_documents=1, timing_repeats=1, decode_tokens=8, loss_chunk=4,
    ),
    "balanced": dict(
        train_contexts="128,256,512", test_contexts="128,256,512,1024,2048",
        block_size=32, features=64,
        train_documents=48, validation_documents=8, test_documents=16, warm_documents=6,
        warm_steps=40, joint_steps=120, eval_every=20,
        timing_documents=2, timing_repeats=2, decode_tokens=24, loss_chunk=8,
    ),
    "extended": dict(
        train_contexts="256,512,1024", test_contexts="256,512,1024,2048,4096",
        block_size=32, features=96,
        train_documents=96, validation_documents=16, test_documents=24, warm_documents=12,
        warm_steps=80, joint_steps=240, eval_every=40,
        timing_documents=3, timing_repeats=3, decode_tokens=32, loss_chunk=8,
    ),
}

if not torch.cuda.is_available() and PROFILE != "smoke":
    raise RuntimeError("Select a GPU runtime or use smoke.")

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/TinyCeNN/qwen35-integrated-v1")
else:
    BASE = Path("/content/qwen35-integrated-v1")

BASE.mkdir(parents=True,exist_ok=True)
run_id = PROFILE + "-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUT = BASE / run_id
LOG = BASE / (run_id + ".log")
RUN = dict(PROFILES[PROFILE], seed=2030)

print(json.dumps(RUN,indent=2))
print("Results:", OUT)


## Preflight

This checks the Qwen3.5 wrapper, Qwen output gate, hybrid cache, checkpoint round-trip, and benchmark entrypoint before doing expensive training.


In [ ]:
# Always synchronize to latest main before testing.
subprocess.run(["git","fetch","origin","main"],cwd=REPO,check=True)
subprocess.run(["git","reset","--hard","origin/main"],cwd=REPO,check=True)
print("Testing source:",subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO,text=True).strip())

env = dict(os.environ, CUDA_VISIBLE_DEVICES="", OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")
r = subprocess.run(
    [sys.executable,"-m","pytest","-q","tests/test_qwen35_integrated_memory.py"],
    cwd=REPO,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
)
print(r.stdout)
if r.returncode:
    raise RuntimeError(f"Qwen3.5 TinyCeNN preflight failed: {r.returncode}")

probe = subprocess.run(
    [sys.executable,str(REPO/"scripts/benchmark_qwen35_integrated_memory.py"),"--help"],
    cwd=REPO,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
)
print(probe.stdout.splitlines()[0] if probe.stdout else "")
if probe.returncode:
    raise RuntimeError("Qwen3.5 benchmark entrypoint failed")
print("✅ Qwen3.5 TinyCeNN preflight passed")


## Train + evaluate

The experiment trains matched controls and CeNN candidates, selects **only a CeNN `cenn_partition` candidate** on validation NLL, then runs held-out test evaluation.


In [ ]:
BENCHMARK = REPO / "scripts/benchmark_qwen35_integrated_memory.py"
cmd = [sys.executable,"-u",str(BENCHMARK),"--base-model",MODEL_ID,"--output-dir",str(OUT)]
for k,v in RUN.items():
    cmd += ["--" + k.replace("_","-"), str(v)]
print("Running:", BENCHMARK.name)
print(" ".join(cmd))

try:
    with LOG.open("w") as log:
        with subprocess.Popen(cmd,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as p:
            for line in p.stdout:
                print(line,end="",flush=True)
                log.write(line); log.flush()
            status = p.wait()
    if status:
        raise RuntimeError(f"Run failed: {status}; inspect {LOG}")
finally:
    if OUT.exists() and LOG.exists():
        shutil.copy2(LOG,OUT/"console.log")
        archive = shutil.make_archive(str(OUT)+"-results","zip",root_dir=OUT)
        print("Archive:",archive)


## Results


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

s = pd.read_csv(OUT/"integrated_summary.csv")
selected = json.loads((OUT/"selection.json").read_text())["selected"]

cols = [c for c in [
    "candidate","context","test_nll","test_perplexity","ppl_ratio","adapted_ppl_ratio",
    "total_cache_ratio","prefill_speedup","decode_speedup","remaining_full_attention_layers",
    "cached_logits_nmse","teacher_cached_logits_nmse","candidate_top1_mismatches",
    "teacher_top1_mismatches","allowed_top1_mismatches","selected_on_validation"
] if c in s.columns]

print("Locked validation selection:",selected)
display(s[cols].sort_values(["candidate","context"]).reset_index(drop=True))

x = s[s.candidate == selected].sort_values("context")
fig, ax = plt.subplots(figsize=(9,4))
ax.plot(x.context,x.ppl_ratio,marker="o",label="PPL / original")
ax.plot(x.context,x.total_cache_ratio,marker="s",label="cache / original")
ax.axhline(1,linestyle="--")
ax.set_xscale("log",base=2)
ax.set_xlabel("Context")
ax.legend()
ax.set_title(selected)
plt.show()

record = json.loads((OUT/"integrated_report.json").read_text())
rec = next(r for r in record["candidates"] if r["candidate"] == selected)
print("Selected layers:", rec["layers"])
print("Remaining original full-attention layers:", rec["remaining_full_attention_layers"])


## Prompt behavior — original Qwen3.5 vs selected CeNN

Greedy decoding is deliberately deterministic here. Token agreement is a trajectory diagnostic, not a semantic quality score.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from tinycenn_lm.qwen35_integrated_memory import restore_student, inference_mode, greedy_generate, native_dtype, new_cache

M = json.loads((OUT/"manifest.json").read_text())
R = json.loads((OUT/"integrated_report.json").read_text())
S = json.loads((OUT/"selection.json").read_text())["selected"]
rec = next(r for r in R["candidates"] if r["candidate"] == S)

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dt = native_dtype(dev)
tok = AutoTokenizer.from_pretrained(MODEL_ID,revision=M["model_revision"])
teacher = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,revision=M["model_revision"],dtype=dt,attn_implementation="sdpa"
).to(dev).eval()
student = restore_student(
    teacher,torch.load(OUT/rec["checkpoint"],map_location="cpu",weights_only=True)
).to(dev).eval()

@torch.no_grad()
def greedy_original(ids,n=48):
    cache = new_cache(teacher)
    logits = teacher(input_ids=ids,past_key_values=cache,use_cache=True).logits[:,-1]
    out = [logits.argmax(-1,keepdim=True)]
    for _ in range(n-1):
        logits = teacher(input_ids=out[-1],past_key_values=cache,use_cache=True).logits[:,-1]
        out.append(logits.argmax(-1,keepdim=True))
    return torch.cat(out,1)

PROMPTS = [
    "What is the capital of Austria?",
    "Explain why the sky appears blue in one short paragraph.",
    "Write a Python function that returns the factorial of n.",
    "A train travels 120 km in 1.5 hours. What is its average speed?",
    "Continue this story: Once upon a time, a small robot discovered a hidden door",
]

for prompt in PROMPTS:
    messages=[{"role":"user","content":prompt}]
    try:
        ids = tok.apply_chat_template(messages,add_generation_prompt=True,tokenize=True,return_tensors="pt").to(dev)
    except Exception:
        ids = tok(prompt,return_tensors="pt")["input_ids"].to(dev)
    A = greedy_original(ids,48)
    with inference_mode(student,str(dt).removeprefix("torch.")):
        B,_ = greedy_generate(student,ids,48)
    n=min(A.shape[1],B.shape[1])
    agreement=float((A[:,:n]==B[:,:n]).float().mean())
    diff=(A[:,:n]!=B[:,:n])[0].nonzero(as_tuple=False)
    first=None if len(diff)==0 else int(diff[0].item())+1
    print("\n"+"="*100)
    print("PROMPT:",prompt)
    print("ORIGINAL:",tok.decode(A[0],skip_special_tokens=False))
    print("CeNN:",tok.decode(B[0],skip_special_tokens=False))
    print(f"Token agreement: {agreement*100:.1f}% | first difference: {first or 'none'}")


## Package + publish to Hugging Face

The publisher creates a reproducible adapter repository with:
- selected CeNN checkpoint
- exact base revision and TinyCeNN source commit
- custom loader/source
- benchmark evidence
- generated model card with the **actual measured results**

Add a Colab secret named `HF_TOKEN` with **write** permission.


In [ ]:
HF_REPO = "vtava/Qwen3.5-0.8B-CeNN-Integrated-V1" # @param {type:"string"}
PUBLISH_TO_HF = True # @param {type:"boolean"}
HF_PRIVATE = False # @param {type:"boolean"}

from google.colab import userdata
try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if PUBLISH_TO_HF and not hf_token:
    raise RuntimeError("Add HF_TOKEN to Colab Secrets with write permission, then rerun this cell.")

PACKAGER = REPO / "scripts/package_qwen35_cenn_hf.py"
publish_cmd = [sys.executable,"-u",str(PACKAGER),"--run-dir",str(OUT),"--repo-id",HF_REPO]
if HF_PRIVATE:
    publish_cmd.append("--private")
if not PUBLISH_TO_HF:
    publish_cmd.append("--no-upload")

env = dict(os.environ)
if hf_token:
    env["HF_TOKEN"] = hf_token
subprocess.run(publish_cmd,cwd=REPO,env=env,check=True)

print("Model card:", OUT/"huggingface_export"/"README.md")
print("HF model:", f"https://huggingface.co/{HF_REPO}")


## Verify uploaded package


In [ ]:
if PUBLISH_TO_HF:
    from huggingface_hub import snapshot_download
    verify_dir = Path(snapshot_download(HF_REPO,token=hf_token))
    required = ["README.md","adapter_config.json","cenn_adapter.pt","load_model.py","integrated_report.json"]
    missing = [name for name in required if not (verify_dir/name).exists()]
    if missing:
        raise RuntimeError(f"HF upload verification missing: {missing}")
    print("✅ Hugging Face package verified:", verify_dir)
    print("✅", f"https://huggingface.co/{HF_REPO}")
